# 1. Загрузка данных

In [1]:
!pip install pandas matplotlib

  Using cached pandas-3.0.5-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached matplotlib-3.11.1-cp312-cp312-win_amd64.whl.metadata (80 kB)
  Using cached numpy-2.5.2-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp312-cp312-win_amd64.whl.metadata (121 kB)
  Using cached kiwisolver-1.5.0-cp312-cp312-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.3.0-cp312-cp312-win_amd64.whl.metadata (9.3 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached pandas-3.0.5-cp312-cp312-win_amd64.whl (9.8 MB)
Using cached matplotlib-3.11.1-cp312-cp312-win_amd64.whl (9.3 MB)
Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl (226 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.63.0-cp312-cp31

In [7]:
!pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]



# Часть 1. Extract & Оптимизация памяти (Data Ingestion)
Как Data Engineers, мы должны контролировать потребление памяти прямо в момент загрузки. 
Посмотрим, как Pandas работает с типами по умолчанию и как мы можем оптимизировать этот процесс.


## Как правильно выбирать типы данных в Pandas: Гайд для Data Engineer

По умолчанию Pandas перестраховывается и выделяет под любые целые числа `int64` (8 байт на число), а под строки — тип `object` (выделяет память под саму строку + указатель на объект Python, что крайне неэффективно).

### 1. Карта выбора целочисленных типов (Integer Downcasting)
В зависимости от максимального и минимального значения в колонке, мы обязаны урезать разрядность, чтобы экономить RAM на больших объемах данных:

| Тип в Pandas | Диапазон значений (Signed) | Для каких колонок подходит? |
| :--- | :--- | :--- |
| **`int8`** | от -128 до 127 | Флаги (`0` или `1`), Класс каюты (`1`, `2`, `3`), Количество родственников (`SibSp`, `Parch`) |
| **`int16`** | от -32 768 до 32 767 | Возраст в днях, Номер отдела, День года |
| **`int32`** | от -2 147 483 648 до 2 147 483 647 | ID пользователей (`PassengerId`), Почтовые индексы, Количество просмотров |
| **`int64`** | Больше $\pm$ 2 млрд | Большие денежные транзакции в центах, Микросекундные Unix-таймстампы |

*DE-совет:* Если в колонке есть пропуски (`NaN`), стандартный `int` упадет с ошибкой. В этом случае используйте nullable-типы Pandas с большой буквы: `Int8`, `Int16` и т.д.

### 2. Когда нужно менять `object` на `category`?
Тип `category` работает по принципу **Dictionary Encoding** (кодирование словарем). Вместо того чтобы дублировать тяжелую строку "Male" 100 000 раз, Pandas создает скрытый словарь `{0: "Male", 1: "Female"}` и хранит в таблице только легковесные числа `0` и `1`.

**Правило инженера (Селективность данных):**
Переводить `object` в `category` стоит ТОЛЬКО тогда, когда количество уникальных значений (кардинальность) значительно меньше общего количества строк (обычно **< 5-10%** от объема датасета).

*   **Идеально для категорий:** Пол (`Sex`), Город/Страна (`Embarked`), Статус заказа, Способ оплаты. (Память падает до 10 раз!).
*   **Запрещено переводить в категории:** Уникальные текстовые поля: ФИО (`Name`), Хэши, Комментарии пользователей, Номера телефонов. Если уникальных значений много, создание словаря только *увеличит* потребление памяти и замедлит джойны.


#### Загрузка данных

In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [33]:
df_json = pd.read_json('titanic.json')
df_json.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


In [35]:
df_xlsx = pd.read_excel('titanic.xlsx', sheet_name=0)
df_xlsx.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


In [38]:
df_titanic  = pd.read_csv('Titanic-Dataset.csv')

In [14]:
df_titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [39]:
df_titanic.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 285.6 KB


In [16]:
optimized_types = {
    'PassengerId': np.int32,
    'Survived': np.int8,
    'Pclass': np.int8,
    'Sex': 'category',
    'SibSp': np.int8,
    'Parch': np.int8,
    'Embarked': 'category'
}

In [40]:
df_titanic = df_titanic.astype(optimized_types)

In [41]:
print("\n--- Состояние после оптимизации ---")
df_titanic.info(memory_usage='deep')


--- Состояние после оптимизации ---
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   PassengerId  891 non-null    int32   
 1   Survived     891 non-null    int8    
 2   Pclass       891 non-null    int8    
 3   Name         891 non-null    str     
 4   Sex          891 non-null    category
 5   Age          714 non-null    float64 
 6   SibSp        891 non-null    int8    
 7   Parch        891 non-null    int8    
 8   Ticket       891 non-null    str     
 9   Fare         891 non-null    float64 
 10  Cabin        204 non-null    str     
 11  Embarked     889 non-null    category
dtypes: category(2), float64(2), int32(1), int8(4), str(3)
memory usage: 169.6 KB


In [42]:
df_titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Эффективная фильтрация данных: Маски vs .query() vs Индексы

В Data Engineering фильтрация — это способ отсечь лишние логи до того, как они пойдут в тяжелые агрегации. В Pandas есть три основных способа фильтрации, и инженер должен знать, когда какой применять.

### 1. Булевы маски (Boolean Masking) — Классика
*   `df[(df['col'] == val) & (df['col2'] > val2)]`
*   **Плюсы**: Работает «из коробки», поддерживает автодополнение кода в IDE.
*   **Минусы**: Ужасно читается, если условий больше трех. Из-за приоритета операторов легко забыть скобки `()` вокруг условий, что вызовет ошибку.

### 2. Метод `.query()` — Инженерный стандарт (Clean Code)
*   `df.query("col == @val and col2 > @val2")`
*   **Плюсы**: Синтаксис похож на `WHERE` в SQL. Код лаконичный и читаемый. Позволяет обращаться к переменным окружения через символ `@`.
*   **Минусы**: Работает чуть медленнее на микро-датасетах, но раскрывается на больших объемах.

### 3. Фильтрация по индексу через `.loc[]` — Самый быстрый способ
*   Если колонка, по которой мы фильтруем — это индекс, Pandas не сканирует всю таблицу целиком (O(N)), а делает быстрый поиск по хэш-таблице за **O(1)**.


In [48]:
df_titanic[
    (df_titanic['Pclass'].isin([1, 2]))
]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S
...,...,...,...,...,...,...,...,...,...,...,...,...
880,881,1,2,"Shelley, Mrs. William (Imanita Parrish Hall)",female,25.0,0,1,230433,26.0000,NaN,S
883,884,0,2,"Banfield, Mr. Frederick James",male,28.0,0,0,C.A./SOTON 34068,10.5000,NaN,S
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S


In [51]:
mask_result = df_titanic[
    (df_titanic['Survived'] == 1) & 
    (df_titanic['Sex'] == 'female') & 
    (df_titanic['Pclass'].isin([1, 2])) &
    (df_titanic['Age'] > 30)
]
mask_result.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S
15,16,1,2,"Hewlett, Mrs. (Mary D Kingcome)",female,55.0,0,0,248706,16.0000,NaN,S
52,53,1,1,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",female,49.0,1,0,PC 17572,76.7292,D33,C


In [52]:
min_age = 30
query_result = df_titanic.query(
    "Survived == 1 and Sex == 'female' and Pclass in [1, 2] and Age > @min_age"
)
query_result.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S


In [53]:
# Проверяем, что результаты идентичны
assert len(mask_result) == len(query_result)
print(f"Найдено записей: {len(query_result)}")

Найдено записей: 76


In [54]:
# --- Производительность: Замеряем скорость на большой таблице ---
large_df = pd.DataFrame({
    'status': np.random.choice(['active', 'inactive', 'pending'], size=1_000_000),
    'value': np.random.randint(0, 100, size=1_000_000)
})

print("\n=== Скорость фильтрации 1 000 000 строк ===")

print("1. Булева маска:")
%timeit large_df[(large_df['status'] == 'active') & (large_df['value'] > 50)]

print("2. Метод .query():")
# На больших объемах numexpr под капотом .query() оптимизирует вычисления в памяти
%timeit large_df.query("status == 'active' and value > 50")


=== Скорость фильтрации 1 000 000 строк ===
1. Булева маска:
145 ms ± 20.3 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
2. Метод .query():
65 ms ± 7.87 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [55]:
# --- Оптимизация через Индексы (Идеально для ID/Ключей) ---
# Если нам нужно часто искать данные по конкретному статусу, делаем его индексом
large_df_indexed = large_df.set_index('status').sort_index()

print("3. Фильтрация по индексу (.loc):")
# Поиск по хэш-таблице / бинарный поиск происходит мгновенно
%timeit large_df_indexed.loc['active']

3. Фильтрация по индексу (.loc):
78.5 μs ± 5.27 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


# Часть 2. Transform: Apply и Работа с Datetime
1. `.apply()`
2. Парсинг дат и извлечение временных признаков.

In [56]:
def get_text_len(text: str) -> int:
    return len(text.split())

In [59]:
df_titanic['name_len'] = df_titanic['Name'].apply(get_text_len)

In [60]:
df_titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,name_len
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,4
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,7
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,3
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,7
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,4


In [62]:
df_titanic['Name'].apply(lambda x: len(x.split()))

0      4
1      7
2      3
3      7
4      4
      ..
886    3
887    4
888    5
889    4
890    3
Name: Name, Length: 891, dtype: int64

In [63]:
df_titanic['Title_Apply'] = df_titanic['Name']\
    .apply(lambda x: x.split(',')[1].split('.')[0].strip())

In [64]:
df_titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,name_len,Title_Apply
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,4,Mr
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,7,Mrs
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,3,Miss
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,7,Mrs
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,4,Mr


#### Datetime

In [66]:
df_orders = pd.read_csv(
    'e_commerce_data/olist_orders_dataset.csv',
    parse_dates=[
        'order_purchase_timestamp', 
        'order_approved_at', 
        'order_delivered_customer_date', 
        'order_estimated_delivery_date'
    ]
)

In [67]:
df_orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  str           
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](4), str(4)
memory usage: 6.1 MB


In [29]:
print(df_orders[['order_purchase_timestamp', 'order_delivered_customer_date']].dtypes)

order_purchase_timestamp         datetime64[us]
order_delivered_customer_date    datetime64[us]
dtype: object


In [30]:
df_orders['delivery_time_days'] = (df_orders['order_delivered_customer_date'] - df_orders['order_purchase_timestamp']).dt.days

In [31]:
df_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0


In [70]:
df_titanic.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
name_len         0
Title_Apply      0
dtype: int64

In [73]:
mean_age = df_titanic['Age'].mean()

In [77]:
df_titanic = df_titanic.fillna({'Age': mean_age, 'Cabin': 'unknown'})

In [78]:
df_titanic.isna().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          0
Embarked       2
name_len       0
Title_Apply    0
dtype: int64

In [79]:
df_items = pd.read_csv('e_commerce_data\olist_order_items_dataset.csv')
df_products = pd.read_csv('e_commerce_data\olist_products_dataset.csv')

<>:1: SyntaxWarning: invalid escape sequence '\o'
<>:2: SyntaxWarning: invalid escape sequence '\o'
<>:1: SyntaxWarning: invalid escape sequence '\o'
<>:2: SyntaxWarning: invalid escape sequence '\o'
C:\Users\shish\AppData\Local\Temp\ipykernel_5240\2702588692.py:1: SyntaxWarning: invalid escape sequence '\o'
  df_items = pd.read_csv('e_commerce_data\olist_order_items_dataset.csv')
C:\Users\shish\AppData\Local\Temp\ipykernel_5240\2702588692.py:2: SyntaxWarning: invalid escape sequence '\o'
  df_products = pd.read_csv('e_commerce_data\olist_products_dataset.csv')


In [81]:
df_merged = pd.merge(df_orders, df_items, on='order_id', how='inner')

In [83]:
df_merged = pd.merge(df_merged, df_products, how='left', on='product_id')

In [89]:
def get_mean(ages):  # [28, 22, 31, ]
    return sum(ages) / len(ages)

In [98]:
df_titanic.groupby(['Sex', ]).agg({'Age': get_mean}).iloc[1]

Age    30.505824
Name: male, dtype: float64

In [94]:
df_merged.to_csv('new_data.csv', index=False)

In [ ]:
df_merged.to_parquet('')